# Nearby Places — proximity features from OpenStreetMap

**Sources:**
- `analysis/poi/osm_raw.json` — raw OpenStreetMap data, pulled once via the
  Overpass API (free, no API key) for Singapore's bounding box
  (1.144, 103.6, 1.472, 104.1), covering 5 categories:
  MRT/train stations, convenience stores, supermarkets, malls, attractions
- `analysis/short_stay_listing_features.csv` — listing `latitude`/`longitude`

This is a **one-time offline pull**, not a live API call per search — the
raw JSON is cached in `analysis/poi/osm_raw.json` so this notebook (and the
web app later) never needs to re-hit Overpass. Distances below are
straight-line (haversine), not walking-route distance — good enough for
"how far / how many nearby," and free. A routing API (e.g. OneMap) would be
the upgrade path later if real walking time is needed.

Produces two files:
- `analysis/poi/singapore_poi.csv` — the cleaned POI list itself (name,
  category, lat/lon) — for **display** ("Nearby: FairPrice, 350m")
- `analysis/short_stay_listing_proximity.csv` — one row per short-stay
  listing with nearest-distance + nearby-count per category — for
  **filtering/ranking** and a quick display summary

In [1]:
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 1. Parse the raw OSM data into a tidy POI table

In [2]:
with open("poi/osm_raw.json") as f:
    osm = json.load(f)

def categorize(tags):
    if tags.get("railway") == "station": return "mrt_station"
    if tags.get("shop") == "convenience": return "convenience_store"
    if tags.get("tourism") == "attraction": return "attraction"
    if tags.get("shop") == "supermarket": return "supermarket"
    if tags.get("shop") == "mall": return "mall"
    return None

rows = []
for el in osm["elements"]:
    tags = el.get("tags", {})
    category = categorize(tags)
    if category is None:
        continue
    # nodes have lat/lon directly; ways/relations use the "center" Overpass computed
    if el["type"] == "node":
        lat, lon = el["lat"], el["lon"]
    else:
        lat, lon = el["center"]["lat"], el["center"]["lon"]
    rows.append({
        "osm_id": el["id"],
        "osm_type": el["type"],
        "category": category,
        "name": tags.get("name", "Unnamed"),
        "latitude": lat,
        "longitude": lon,
    })

poi = pd.DataFrame(rows)
print(f"Total POIs parsed: {len(poi):,}")
poi["category"].value_counts()

Total POIs parsed: 2,000


category
convenience_store    743
supermarket          559
attraction           356
mall                 249
mrt_station           93
Name: count, dtype: int64

In [3]:
poi.to_csv("poi/singapore_poi.csv", index=False)
print(f"Saved {len(poi):,} POIs to analysis/poi/singapore_poi.csv")
poi.head()

Saved 2,000 POIs to analysis/poi/singapore_poi.csv


,osm_id,osm_type,category,name,latitude,longitude
0,206477134,node,mrt_station,Aljunied,1.316,103.883
1,206493462,node,mrt_station,Kallang,1.311,103.871
2,206529415,node,mrt_station,Yio Chu Kang,1.382,103.845
3,207725957,node,mrt_station,Bedok,1.324,103.930
4,208526059,node,mrt_station,Redhill,1.290,103.817


## 2. Compute nearest-distance + nearby-count per listing

Haversine distance (great-circle, in km) from every short-stay listing to
every POI, vectorized with numpy. For each category: distance to the
nearest one, and how many fall within 1km (a comfortable walking radius).

In [4]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius, km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

listings = pd.read_csv("short_stay_listing_features.csv", usecols=["id", "latitude", "longitude"])
listings = listings.dropna(subset=["latitude", "longitude"])
print(f"Listings with coordinates: {len(listings):,}")

Listings with coordinates: 1,483


In [5]:
WALK_RADIUS_KM = 1.0
CATEGORIES = ["mrt_station", "convenience_store", "supermarket", "mall", "attraction"]

result = {"listing_id": listings["id"].values}

for cat in CATEGORIES:
    cat_poi = poi[poi["category"] == cat]
    # distance matrix: (n_listings, n_pois_in_category)
    dists = haversine_km(
        listings["latitude"].values[:, None], listings["longitude"].values[:, None],
        cat_poi["latitude"].values[None, :], cat_poi["longitude"].values[None, :],
    )
    result[f"{cat}_nearest_km"] = dists.min(axis=1).round(3)
    result[f"{cat}_count_within_{WALK_RADIUS_KM:g}km"] = (dists <= WALK_RADIUS_KM).sum(axis=1)

proximity = pd.DataFrame(result)
proximity.head()

,listing_id,mrt_station_nearest_km,mrt_station_count_within_1km,convenience_store_nearest_km,convenience_store_count_within_1km,supermarket_nearest_km,supermarket_count_within_1km,mall_nearest_km,mall_count_within_1km,attraction_nearest_km,attraction_count_within_1km
0,4499317,2.515,0,0.092,48,0.129,17,0.142,29,0.182,7
1,4541183,3.547,0,0.007,42,0.310,12,0.302,10,0.828,3
2,4973227,3.547,0,0.007,42,0.310,12,0.302,10,0.828,3
3,5052014,2.346,0,0.139,21,0.259,7,0.263,5,0.685,2
4,5233415,2.389,0,0.093,24,0.267,7,0.350,5,0.533,3


## 3. Sanity check

In [6]:
proximity.describe()

,listing_id,mrt_station_nearest_km,mrt_station_count_within_1km,convenience_store_nearest_km,convenience_store_count_within_1km,supermarket_nearest_km,supermarket_count_within_1km,mall_nearest_km,mall_count_within_1km,attraction_nearest_km,attraction_count_within_1km
count,"1,483.000","1,483.000","1,483.000","1,483.000","1,483.000","1,483.000","1,483.000","1,483.000","1,483.000","1,483.000","1,483.000"
mean,"1,011,243,906,401,041,408.000",2.424,0.084,0.207,27.482,0.302,11.657,0.366,12.829,0.689,3.039
std,"581,061,680,203,224,832.000",0.757,0.428,0.228,17.067,0.297,5.007,0.314,9.164,0.496,2.645
min,"4,499,317.000",0.038,0.000,0.002,0.000,0.005,0.000,0.009,0.000,0.026,0.000
25%,"708,752,130,651,192,960.000",2.061,0.000,0.083,16.000,0.152,9.000,0.189,6.000,0.341,1.000
50%,"1,267,274,248,707,779,840.000",2.575,0.000,0.169,22.000,0.232,12.000,0.309,9.000,0.651,2.000
75%,"1,416,838,000,919,165,952.000",2.827,0.000,0.275,44.000,0.351,13.000,0.434,21.000,0.869,4.000
max,"1,716,877,869,359,547,648.000",5.178,5.000,3.182,69.000,3.502,25.000,3.223,34.000,3.996,13.000


## 4. Save

In [7]:
proximity.to_csv("short_stay_listing_proximity.csv", index=False)
print(f"Saved {len(proximity):,} rows to analysis/short_stay_listing_proximity.csv")

Saved 1,483 rows to analysis/short_stay_listing_proximity.csv
